In [ ]:
from googleapiclient.discovery import build
import csv
from datetime import datetime, timezone
import time

In [ ]:
# 설정
API_KEY = "여기에_발급받은_API_KEY_작성"
OUTPUT_FILE = "youtube_public_playlists_full.csv"
MAX_COMMENTS_PER_VIDEO = 500

# 수집 대상 공개 채널 (채널 ID)
CHANNELS = {
    "채널 이름": "채널 ID"
}

# YouTube API 빌드
youtube = build("youtube", "v3", developerKey=API_KEY)

In [ ]:
# 공개 채널 재생목록 가져오기
def fetch_playlists(channel_id):
    playlists = []
    next_page_token = None
    while True:
        res = youtube.playlists().list(
            part="id,snippet",
            channelId=channel_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for pl in res.get("items", []):
            playlists.append({
                "playlist_id": pl["id"],
                "title": pl["snippet"]["title"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return playlists

In [ ]:
# 재생목록 안 영상 가져오기
def fetch_playlist_videos(playlist_id):
    videos = []
    next_page_token = None
    while True:
        res = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for item in res.get("items", []):
            videos.append({
                "video_id": item["snippet"]["resourceId"]["videoId"],
                "title": item["snippet"]["title"],
                "description": item["snippet"]["description"],
                "publish_time": item["snippet"]["publishedAt"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return videos

In [ ]:
# 영상 통계 조회
def fetch_video_stats(video_id):
    resp = youtube.videos().list(
        part="snippet,statistics",
        id=video_id
    ).execute()

    if not resp["items"]:
        return None

    item = resp["items"][0]

    return {
        "upload_time": item["snippet"]["publishedAt"], # :point_left: 이게 진짜 업로드 시각
        "viewCount": item["statistics"].get("viewCount","0"),
        "likeCount": item["statistics"].get("likeCount","0"),
        "commentCount": item["statistics"].get("commentCount","0")
    }

In [ ]:
# 댓글 수집 + 좋아요
def fetch_comments(video_id):
    all_comments = []
    next_page_token = None

    while True:
        res = youtube.commentThreads().list(
            part="snippet,replies",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText",
            pageToken=next_page_token
        ).execute()

        for thread in res.get("items", []):
            top = thread["snippet"]["topLevelComment"]["snippet"]
            all_comments.append({
                "text": top["textDisplay"],
                "likeCount": top.get("likeCount", 0)
            })

            if "replies" in thread:
                for reply in thread["replies"]["comments"]:
                    r = reply["snippet"]
                    all_comments.append({
                        "text": r["textDisplay"],
                        "likeCount": r.get("likeCount", 0)
                    })

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

    # 좋아요 내림차순 정렬 후 최대 500개만
    all_comments.sort(key=lambda x: x["likeCount"], reverse=True)
    return all_comments[:500]

In [ ]:
# 영상 + 댓글 + 통계 처리
def process_video(video, playlist_title):
    video_id = video["video_id"]
    publish_utc = datetime.fromisoformat(video["publish_time"].replace("Z","+00:00")).strftime("%Y-%m-%d %H:%M:%S")

    stats = fetch_video_stats(video_id)
    comments = fetch_comments(video_id)

    rows = []
    if not comments:
        rows.append([
            playlist_title, video_id, video["title"], video["description"], stats["upload_time"],
            stats["viewCount"], stats["likeCount"], stats["commentCount"],
            "", ""])
    else:
        for c in comments:
            rows.append([
                playlist_title, video_id, video["title"], video["description"], stats["upload_time"],
                stats["viewCount"], stats["likeCount"], stats["commentCount"],
                c["text"], c["likeCount"]
            ])
    return rows

In [ ]:
# 메인 수집
def collect_youtube_public_playlists():
    total_processed = 0
    start_time = time.time()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "playlist_title",
            "video_id",
            "title",
            "description",
            "publish_time_utc",
            "viewCount",
            "likeCount",
            "commentCount",
            "comment_text",
            "comment_likeCount"
        ])

        for channel_name, channel_id in CHANNELS.items():
            print(f"\n===== 채널 수집 시작: {channel_name} =====")
            playlists = fetch_playlists(channel_id)
            print(f"총 {len(playlists)}개의 재생목록 발견")

            for pl in playlists:
                playlist_title = pl["title"]
                print(f"\n----- 재생목록 수집 시작: {playlist_title} -----")
                videos = fetch_playlist_videos(pl["playlist_id"])
                print(f"총 {len(videos)}개의 영상 발견")

                for video in videos:
                    rows = process_video(video, playlist_title)
                    for row in rows:
                        writer.writerow(row)
                    total_processed += 1

                    # 진행률 및 ETA
                    progress_ratio = total_processed / max(len(videos),1)
                    elapsed_time = time.time() - start_time
                    avg_time = elapsed_time / max(total_processed,1)
                    remaining_time = avg_time * (len(videos) - total_processed)

                    if total_processed % 5 == 0:
                        print(
                            f"[{playlist_title}] 저장 중 영상 ID: {rows[0][1]} | "
                            f"게시글 UTC: {rows[0][4]} | "
                            f"진행률: {progress_ratio*100:.2f}% | "
                            f"총 처리: {total_processed}개 | "
                            f"평균: {avg_time:.2f}초/개 | "
                            f"예상 남은 시간: {remaining_time/60:.1f}분"
                        )

                print(f"----- 재생목록 수집 완료: {playlist_title} -----")

            print(f"===== 채널 수집 완료: {channel_name} =====")

    print("\n===== 전체 YouTube 공개 재생목록 수집 완료 =====")


In [ ]:
# 실행
collect_youtube_public_playlists()